# Seiche audited direct-OFR research snapshot

**Publication status: draft, not submitted. No DOI has been assigned.**

This notebook is designed for Colab, Binder, JupyterLab, or local Jupyter. It reads only two public CSVs pinned to a Seiche Git commit, verifies their hashes, and fails closed unless they contain exactly the ten rights-audited Office of Financial Research series and 11,163 observations. It never reads the Seiche runtime cache and never mixes in a FRED-fetched value.

Sources: [OFR API](https://www.financialresearch.gov/short-term-funding-monitor/api/), [Repo Markets release](https://www.financialresearch.gov/short-term-funding-monitor/datasets/repo/), [Money Market Fund release](https://www.financialresearch.gov/short-term-funding-monitor/datasets/mmf/), and [OFR legal notices](https://www.financialresearch.gov/legal-notices/). The legal notice explains the federal-work boundary, requests credit, and warns that separately copyrighted material still requires permission. This is an engineering rights review, not legal advice or OFR endorsement.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict
from datetime import date

COMMIT = "93e83bbc592098fc2f6465ffb49c5e872d61c018"
RAW_ROOT = f"https://raw.githubusercontent.com/beepboop2025/seiche/{COMMIT}/integrations/datacommons/input/observations"
FILES = {
    "ofr_repo_markets.csv": {
        "rows": 10_489,
        "sha256": "307ae6ad5bbe8653c3bd4abf63449d4229b0fbba1ee66014d91f813e866d3a4a",
    },
    "ofr_money_market_funds.csv": {
        "rows": 674,
        "sha256": "1d0975b69dcb6f3465e957679b21bacae02695145b1e77563dae3258d8b524cd",
    },
}
EXPECTED_VARIABLES = {
    "seiche/OfrDvpOvernightOpenRepo_Mean_InterestRate_FinancialInstrument",
    "seiche/OfrDvpRepoTotal_Sum_Amount_FinancialTransaction",
    "seiche/OfrGcfOvernightOpenRepo_Mean_InterestRate_FinancialInstrument",
    "seiche/OfrGcfOvernightOpenRepo_Sum_Amount_FinancialTransaction",
    "seiche/OfrMoneyMarketFundRepoFederalReserve_Sum_Amount_Investment",
    "seiche/OfrMoneyMarketFundRepoFicc_Sum_Amount_Investment",
    "seiche/OfrMoneyMarketFundRepoTotal_Sum_Amount_Investment",
    "seiche/OfrMoneyMarketFundTotal_Sum_Amount_Investment",
    "seiche/OfrTriPartyOvernightOpenRepo_Mean_InterestRate_FinancialInstrument",
    "seiche/OfrTriPartyRepoTotal_Sum_Amount_FinancialTransaction",
}
EXPECTED_COLUMNS = ["entity", "variable", "date", "value", "unit", "measurementMethod", "observationPeriod"]

In [ ]:
def fetch_and_verify(name: str, receipt: dict) -> list[dict[str, str]]:
    url = f"{RAW_ROOT}/{name}"
    request = urllib.request.Request(url, headers={"User-Agent": "seiche-research-notebook/1.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        raw = response.read(2_000_001)
    if len(raw) > 2_000_000:
        raise ValueError(f"{name}: response exceeds the 2 MB notebook limit")
    observed_hash = hashlib.sha256(raw).hexdigest()
    if observed_hash != receipt["sha256"]:
        raise ValueError(f"{name}: SHA-256 changed; stop for review")
    reader = csv.DictReader(io.StringIO(raw.decode("utf-8")))
    rows = list(reader)
    if list(reader.fieldnames or []) != EXPECTED_COLUMNS or len(rows) != receipt["rows"]:
        raise ValueError(f"{name}: schema or row count changed; stop for review")
    return rows

frames = [fetch_and_verify(name, receipt) for name, receipt in FILES.items()]
observations = [row for frame in frames for row in frame]
if len(observations) != 11_163:
    raise ValueError("combined row count changed; stop for review")
if {row["variable"] for row in observations} != EXPECTED_VARIABLES:
    raise ValueError("series allowlist changed; stop for review")
if {row["entity"] for row in observations} != {"country/USA"}:
    raise ValueError("entity scope changed; stop for review")
keys = {(row["entity"], row["variable"], row["date"]) for row in observations}
if len(keys) != len(observations):
    raise ValueError("duplicate observation keys found; stop for review")
for row in observations:
    date.fromisoformat(row["date"])
    if not math.isfinite(float(row["value"])):
        raise ValueError("non-finite observation found; stop for review")
print(f"Validated {len(observations):,} direct-OFR observations across {len(EXPECTED_VARIABLES)} series.")

## Preserve native clocks

Repo rows are daily preliminary observations; money-market-fund rows are monthly revised complete-series observations. The summary below reports each series' own first and last observation. It does not forward-fill one publisher clock into another or equate notebook execution time with evidence time.

In [ ]:
by_series = defaultdict(list)
for row in observations:
    by_series[row["variable"]].append(row)
series_receipt = []
for variable, rows in sorted(by_series.items()):
    series_receipt.append({
        "variable": variable,
        "unit": rows[0]["unit"],
        "measurementMethod": rows[0]["measurementMethod"],
        "rows": len(rows),
        "first_observation": min(row["date"] for row in rows),
        "last_observation": max(row["date"] for row in rows),
    })
for receipt in series_receipt:
    print(f"{receipt['variable']:<86} {receipt['rows']:>5}  {receipt['first_observation']}  {receipt['last_observation']}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Optional chart skipped: install matplotlib to render it. All validation and summaries above are standard-library only.")
else:
    rate_variables = sorted(v for v in EXPECTED_VARIABLES if "InterestRate" in v)
    fig, ax = plt.subplots(figsize=(11, 5))
    for variable in rate_variables:
        rows = sorted(by_series[variable], key=lambda row: row["date"])
        ax.plot([date.fromisoformat(row["date"]) for row in rows], [float(row["value"]) for row in rows], linewidth=1, label=variable.split("/", 1)[-1].split("_Mean", 1)[0])
    ax.set_title("Audited direct-OFR repo rates (native observation dates)")
    ax.set_ylabel("Percent")
    ax.set_xlabel("OFR observation date")
    ax.legend(fontsize=8)
    fig.tight_layout()

## Optional live citation and clock receipt

The dataset above is immutable and commit-pinned. The following independent request reads Seiche's public, bounded world-markets contract and prints only its citation, clocks, and scope. A current response does not advance any source as-of clock; an unavailable service stays explicitly unavailable.

In [ ]:
endpoint = "https://api.seiche.info/api/v2/world-markets?" + urllib.parse.urlencode({"section": "sources"})
request = urllib.request.Request(endpoint, headers={"Accept": "application/json", "User-Agent": "seiche-research-notebook/1.0"})
try:
    with urllib.request.urlopen(request, timeout=15) as response:
        raw = response.read(2_000_001)
    if len(raw) > 2_000_000:
        raise ValueError("world-markets response exceeds the 2 MB notebook limit")
    world = json.loads(raw)
    if world.get("schema") != "seiche.world-markets.v1" or world.get("selection") != "sources":
        raise ValueError("unexpected world-markets contract")
    live_receipt = {"status": world.get("status"), "citation": world.get("citation"), "clocks": world.get("clocks"), "scope": world.get("scope")}
except urllib.error.HTTPError as exc:
    live_receipt = {"status": "unavailable", "http_status": exc.code, "retry_after": exc.headers.get("Retry-After"), "reason": "No live evidence was substituted."}
live_receipt